# Extract Word to Dataframe

In [1]:
from docx import Document
import pandas as pd
import os

In [2]:
def extract_text(file_path):
    doc = Document(file_path)
    text = []
    for para in doc.paragraphs:
        text.append(para.text)
    return "\n".join(text)

# Read and process all .docx files in the specified folder
folder_path = "dataset"
data = []
for filename in os.listdir(folder_path):
    if filename.endswith(".docx"):
        file_path = os.path.join(folder_path, filename)
        print(f"Processing: {filename}")
        
        try:
            text = extract_text(file_path)
            data.append({
                "filename": filename,
                "text": text
            })
        except Exception as e:
            print(f"  [ERROR] {filename}: {e}")

# Create a DataFrame and save to CSV
df = pd.DataFrame(data)
df["ID"] = df.index + 1
print(df.head())
print(f"Total: {len(df)} dokumen")

output_csv = "resumes_text.csv"
df.to_csv(output_csv, index=False)

Processing: Abiral_Pandey_Fullstack_Java.docx
Processing: Achyuth Resume_8.docx
Processing: Adelina_Erimia_PMP1.docx
Processing: Adhi Gopalam - SM.docx
Processing: AjayKumar.docx
Processing: Akhil.profile.docx
Processing: Akhil_Sr BSA.docx
Processing: Alekhya Resume.docx
Processing: Amar Sr BSA.docx
Processing: Ami Jape.docx
Processing: Amrinder Business Analyst.docx
Processing: Amulya Komatineni.docx
Processing: Anil Krishna Mogalaturthi.docx
Processing: AnilAgarwal.docx
Processing: Anudeep N_Sr Java Developer.docx
Processing: Ashok Jayakumar - PM.docx
Processing: Ashwini J2EE Developer.docx
Processing: Atul_Mathur_Resume.docx
Processing: Avathika BA-Healthcare_.docx
Processing: avinash G.docx
Processing: B Shaker-Sr BSA-Scrum Master .docx
Processing: B Suresh Kumar_Project Manager_1.docx
Processing: BA - Abhishek.docx
Processing: BA - Navneet.docx
Processing: BA Kiran.docx
Processing: BA with INV.docx
Processing: Balaji Gopalakrishnan Project Manager.docx
Processing: Balakrishna Suda

# Cleaning

In [3]:
df

,filename,text,ID
0,Abiral_Pandey_Fullstack_Java.docx,Name: Abiral Pandey\nEmail: abiral.pandey88@gm...,1
1,Achyuth Resume_8.docx,Achyuth\n540-999-8048\nachyuth.java88@gmail.co...,2
2,Adelina_Erimia_PMP1.docx,"Adelina Erimia, PMP, Six Sigma Green Belt, SMC...",3
3,Adhi Gopalam - SM.docx,Adhi Gopalam\nadhigopalam@gmail.com\n281-212-3...,4
4,AjayKumar.docx,Ajay Kumar (CSM)\t \t\t Email/Skype: a...,5
...,...,...,...
219,Vishnu Java dev.docx,VISHNU J\nEmail to jammigumpulavishnu452@gmail...,220
220,Vivek Joshi_CV.docx,"\nVivek Joshi\nBusiness Analyst\nBellevue, WA\...",221
221,Vivek.BSA.docx,VIVEK SAGAR ...,222
222,Yohan BSA.docx,"YOHAN \nSr. Business Analyst\n\nVersatile, eff...",223


In [4]:
# Drop missing values
df.dropna(inplace=True)

In [5]:
import re
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = text.lower()                          # 1. lowercase
    text = re.sub(r'[^a-zA-Z\s]', '', text)     # 2. hapus karakter aneh
    text = re.sub(r'\s+', ' ', text).strip()     # 3. hapus whitespace berlebih
    text = " ".join(word for word in text.split() 
                    if word not in stop_words)    # 4. hapus stopwords
    return text

df["text_clean"] = df["text"].apply(clean_text)

# Segmentation

In [6]:
SECTIONS = {
    "objective" : [
        "objective", "summary", "profile", "about me", "about",
        "professional summary", "career objective", "personal statement",
        "career summary", "professional profile", "overview", "introduction"
    ],
    "experience" : [
        "experience", "work experience", "work history", "employment",
        "employment history", "professional experience", "career history",
        "relevant experience", "internship", "internship experience",
        "project experience", "working experience", "job history"
    ],
    "education" : [
        "education", "academic background", "academic history",
        "academic qualification", "educational background", "qualification",
        "certifications", "courses", "training", "degrees", "academics"
    ],
    "skills" : [
        "skills", "technical skills", "core competencies", "competencies",
        "expertise", "key skills", "technologies", "tools", "proficiencies",
        "technical proficiencies", "capabilities", "strengths",
        "skill set", "my skillset", "programming skills",
        "soft skills", "hard skills"
    ]
}

SCHEMA_MAP = {
    keyword: section
    for section, keywords in SECTIONS.items()
    for keyword in keywords
}

def detect_header(line):
    clean = line.strip().lower().rstrip(":").strip()
    if not clean or len(clean.split()) > 5:
        return None
    return SCHEMA_MAP.get(clean, None)

def segmentation(text):
    schema = {
        "objective"  : "",
        "experience" : "",
        "education"  : "",
        "skills"     : "",
        "others"     : ""
    }

    if not isinstance(text, str) or text == "":
        return schema

    lines = text.split("\n")
    current_section = "others"

    for line in lines:
        header = detect_header(line)
        if header:
            current_section = header
        else:
            schema[current_section] += line + " "

    schema = {k: v.strip() for k, v in schema.items()}
    return schema

for section in ["objective", "experience", "education", "skills", "others"]:
    df[section] = df["text"].apply(
        lambda x: segmentation(x)[section] if isinstance(x, str) else ""
    )

print(f"Skills terdeteksi    : {(df['skills'] != '').sum()}/{len(df)}")
print(f"Experience terdeteksi: {(df['experience'] != '').sum()}/{len(df)}")
print(f"Education terdeteksi : {(df['education'] != '').sum()}/{len(df)}")
print(f"Objective terdeteksi : {(df['objective'] != '').sum()}/{len(df)}")

print("\nSample hasil:")
print("SKILLS    :", df["skills"][0][:150])
print("EXPERIENCE:", df["experience"][0][:150])
print("EDUCATION :", df["education"][0][:150])
print("OBJECTIVE :", df["objective"][0][:150])

Skills terdeteksi    : 68/224
Experience terdeteksi: 172/224
Education terdeteksi : 108/224
Objective terdeteksi : 175/224

Sample hasil:
SKILLS    : Programming Languages: Java/J2EE, PL/SQL, Unix Shell Scripts Java/J2EE Technologies: JavaBeans, collections, Servlets, JSP, JDBC, JNDI, RMI, EJB Frame
EXPERIENCE: CVS, Woonsocket, Rhode Island                                 Full Stack Java Developer April 2016 – Present Responsibilities: Involved in various sta
EDUCATION : Bachelor of Computer Science – University of North Texas, Denton, Texas
OBJECTIVE : Dynamic individual with 6 years of software development experience in design, development, deployment, maintenance, production and support of web - ba


In [7]:
df

,filename,text,ID,text_clean,objective,experience,education,skills,others
0,Abiral_Pandey_Fullstack_Java.docx,Name: Abiral Pandey\nEmail: abiral.pandey88@gm...,1,name abiral pandey email abiralpandeygmailcom ...,Dynamic individual with 6 years of software de...,"CVS, Woonsocket, Rhode Island ...",Bachelor of Computer Science – University of N...,"Programming Languages: Java/J2EE, PL/SQL, Unix...",Name: Abiral Pandey Email: abiral.pandey88@gma...
1,Achyuth Resume_8.docx,Achyuth\n540-999-8048\nachyuth.java88@gmail.co...,2,achyuth achyuthjavagmailcom objective around y...,Around 8 years of strong software experience i...,Client: Capital One\t\t\t\t\t\t\t\t\t\t ...,,,Achyuth 540-999-8048 achyuth.java88@gmail.com
2,Adelina_Erimia_PMP1.docx,"Adelina Erimia, PMP, Six Sigma Green Belt, SMC...",3,adelina erimia pmp six sigma green belt smc er...,,,,"▪ Proficient with MS Project, PPM, Smartsheet,...","Adelina Erimia, PMP, Six Sigma Green Belt, SMC..."
3,Adhi Gopalam - SM.docx,Adhi Gopalam\nadhigopalam@gmail.com\n281-212-3...,4,adhi gopalam adhigopalamgmailcom certified scr...,,"Black box Network, Pittsburgh, PA\t\t\t\t\t\t\...","MBA from Vinayaka Mission, Salem, Tamilnadu, I...",,Adhi Gopalam adhigopalam@gmail.com 281-212-359...
4,AjayKumar.docx,Ajay Kumar (CSM)\t \t\t Email/Skype: a...,5,ajay kumar csm emailskype ajaydtgmailcom mob p...,"14 Years IT experience as an Architect, Soluti...","UHG (Optum Insight), MN, USA - Oct 2015 to Til...",,,Ajay Kumar (CSM)\t \t\t Email/Skype: a...
...,...,...,...,...,...,...,...,...,...
219,Vishnu Java dev.docx,VISHNU J\nEmail to jammigumpulavishnu452@gmail...,220,vishnu j email jammigumpulavishnugmailcom cont...,Eight Years of professional IT experience as a...,"Equifax, St.Louis, Missouri\t ...","Bachelor’s in Engineering, JNTUK University, I...",,VISHNU J Email to jammigumpulavishnu452@gmail....
220,Vivek Joshi_CV.docx,"\nVivek Joshi\nBusiness Analyst\nBellevue, WA\...",221,vivek joshi business analyst bellevue wa busin...,,System Analyst –I Net Soft Inc(Sept 2017 to ti...,Bachelor of Engineering – 2005 batch,"Agile (Rally, TDP, JIRA),Waterfall, SDLC, QA M...","Vivek Joshi Business Analyst Bellevue, WA Bus..."
221,Vivek.BSA.docx,VIVEK SAGAR ...,222,vivek sagar sr business system analyst profess...,Senior Business Systems Analyst with 8+ years’...,,,DOMAIN KNOWLEDGE PROFFESSIONAL EXPERIENCE HSB...,VIVEK SAGAR ...
222,Yohan BSA.docx,"YOHAN \nSr. Business Analyst\n\nVersatile, eff...",223,yohan sr business analyst versatile effective ...,Competent experience from working on different...,Sr. Business Systems Analyst NXP Semiconductor...,,,"YOHAN Sr. Business Analyst Versatile, effect..."


# Preprocessing

In [8]:
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from nltk import pos_tag

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt_tab')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\grace\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\grace\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\grace\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\grace\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [10]:
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

lemmatizer = WordNetLemmatizer()

def process_text(text):
    if not isinstance(text, str) or text.strip() == "":
        return "", ""

    # Tokenisasi
    words = word_tokenize(text)

    # POS tagging
    pos_tags = nltk.pos_tag(words)

    # Lemmatization + simpan POS
    lemmatized_words = []
    pos_tagged_words = []

    for word, pos in pos_tags:
        lemma = lemmatizer.lemmatize(word, get_wordnet_pos(pos))
        lemmatized_words.append(lemma)
        pos_tagged_words.append(f"{word}/{pos}")

    return " ".join(lemmatized_words), " ".join(pos_tagged_words)

lemmatized_texts = []
pos_tagged_texts = []

for text in df["text_clean"]:
    lemma, pos = process_text(text)
    lemmatized_texts.append(lemma)
    pos_tagged_texts.append(pos)

df["text_lemmatized"] = lemmatized_texts
df["text_pos_tagged"] = pos_tagged_texts

print("CLEAN       :", df["text_clean"][0][:300])
print("\nLEMMATIZED  :", df["text_lemmatized"][0][:300])
print("\nPOS TAGGED  :", df["text_pos_tagged"][0][:300])

CLEAN       : name abiral pandey email abiralpandeygmailcom phone current location woonsocket rhode island visa status us citizen summary dynamic individual years software development experience design development deployment maintenance production support web based clientserver business applications using oop jav

LEMMATIZED  : name abiral pandey email abiralpandeygmailcom phone current location woonsocket rhode island visa status u citizen summary dynamic individual year software development experience design development deployment maintenance production support web base clientserver business application use oop javajee t

POS TAGGED  : name/NN abiral/JJ pandey/NN email/NN abiralpandeygmailcom/IN phone/NN current/JJ location/NN woonsocket/NN rhode/NN island/NN visa/NN status/NN us/PRP citizen/VBP summary/JJ dynamic/JJ individual/JJ years/NNS software/NN development/NN experience/NN design/NN development/NN deployment/NN maintenance


# NER

In [11]:
import spacy
nlp = spacy.load("en_core_web_sm")

# Skill list untuk matching
SKILL_LIST = [
    "python", "java", "aws", "javascript", "sql", "excel",
    "sap", "tensorflow", "pytorch", "docker", "kubernetes",
    "react", "nodejs", "mongodb", "postgresql", "git",
    "html", "css", "php", "swift", "kotlin", "flutter",
    "machine learning", "deep learning", "nlp", "tableau",
    "power bi", "spark", "hadoop", "azure", "gcp"
]

def extract_ner(text):
    if not isinstance(text, str) or text == "":
        return [], [], [], []

    doc = nlp(text)

    # Ekstrak entitas dari spaCy
    organisations = [ent.text for ent in doc.ents if ent.label_ == "ORG"]
    dates         = [ent.text for ent in doc.ents if ent.label_ == "DATE"]
    persons       = [ent.text for ent in doc.ents if ent.label_ == "PERSON"]

    # Ekstrak skills pakai noun + skill list
    nouns  = [token.text.lower() for token in doc 
              if token.pos_ in ["NOUN", "PROPN"]]
    skills = list(set(word for word in nouns if word in SKILL_LIST))

    return organisations, dates, persons, skills

# Apply ke DataFrame
orgs, dates, persons, skills = [], [], [], []

for text in df["text_lemmatized"]:
    o, d, p, s = extract_ner(text)
    orgs.append(o)
    dates.append(d)
    persons.append(p)
    skills.append(s)

df["ner_organisations"] = orgs
df["ner_dates"]         = dates
df["ner_persons"]       = persons
df["ner_skills"]        = skills

# Cek hasil
print("SKILLS :", df["ner_skills"][0])
print("ORGS   :", df["ner_organisations"][0])
print("DATES  :", df["ner_dates"][0])

SKILLS : ['git', 'mongodb', 'html', 'aws', 'java', 'javascript', 'excel', 'sql', 'postgresql']
ORGS   : ['jsp', 'jaxrs jersey framework soap use', 'ibm', 'oracle weblogic', 'xp aix sun solaris', 'ade factory pattern build', 'ibm', 'ibm', 'ibm', 'ibm', 'aix sun solaris', 'ibm', 'jsp', 'jsp', 'jsp', 'jsp', 'jsp', 'jsp', 'jsp', 'oracle html xml']
DATES  : ['spring', 'april', 'december march', 'spring', 'november december', 'xslt', 'may october', 'january april']
